# Filter execution score annotations



In [ ]:
from pathlib import Path
import copy
import json

PROJECT_ROOT = Path("..").resolve()
INPUT_PATH = PROJECT_ROOT / "notebooks" / "data" / "annotations-export.json"

# Set to an integer to keep only the first n tasks. Use None to keep all tasks.
n = None

suffix = "all" if n is None else f"first-{n}"
OUTPUT_PATH = (
    PROJECT_ROOT
    / "notebooks"
    / "data"
    / f"annotations-export-without-execution-score-{suffix}.json"
)

REMOVE_RESULTS = {
    ("execution_score", "rating"),
}
REMOVE_TASK_KEYS = ("drafts", "predictions")


In [ ]:
with INPUT_PATH.open() as f:
    annotations = json.load(f)

selected_annotations = annotations[:] if n is None else annotations[:n]
filtered_annotations = copy.deepcopy(selected_annotations)

removed_results = 0
removed_drafts = 0
removed_predictions = 0
touched_annotations = 0

for task in filtered_annotations:
    removed_drafts += len(task.get("drafts", []))
    removed_predictions += len(task.get("predictions", []))
    for task_key in REMOVE_TASK_KEYS:
        task.pop(task_key, None)

    for annotation in task.get("annotations", []):
        original_result = annotation.get("result", [])
        filtered_result = [
            item
            for item in original_result
            if not (
                (item.get("from_name"), item.get("type")) in REMOVE_RESULTS
            )
        ]

        removed_here = len(original_result) - len(filtered_result)
        if removed_here:
            removed_results += removed_here
            touched_annotations += 1

        annotation["result"] = filtered_result
        if "result_count" in annotation:
            annotation["result_count"] = len(filtered_result)

remaining_removed_results = sum(
    1
    for task in filtered_annotations
    for annotation in task.get("annotations", [])
    for item in annotation.get("result", [])
    if (item.get("from_name"), item.get("type")) in REMOVE_RESULTS
)

print(
    {
        "loaded_tasks": len(annotations),
        "selected_tasks": len(selected_annotations),
        "removed_drafts": removed_drafts,
        "removed_predictions": removed_predictions,
        "touched_annotations": touched_annotations,
        "removed_results": removed_results,
        "remaining_removed_results": remaining_removed_results,
    }
)


In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

with OUTPUT_PATH.open("w") as f:
    json.dump(filtered_annotations, f, indent=2)

print(OUTPUT_PATH)
